# Statistics and Graphics Applied to the Lecture 6 Count Matrix

- The deck `lecture-09_slides.qmd` explains the statistics (2–6) and the graphics (7–12). This notebook **applies** them to real data.
- Data: the Lecture 6 count matrix — 39,376 genes × 12 samples. 2 cell lines (`PC9`, `PC9GR`) × 2 treatments (`Control`, `Apatinib`) × 3 replicates.

## Get the data: `counts_annotated.tsv`

Every code cell below reads one file: **`counts_annotated.tsv`** — 39,376 genes × 12 samples. Columns: `GeneID`, `Symbol`, then `PC9_Control_1 … PC9GR_Apatinib_3`. The file is **not** in the Colab runtime, so get it before running anything.

**Source (Google Drive):** [counts_annotated.tsv](https://drive.google.com/file/d/1Ym9S2vvbuTbAWq2mKFlQfLxLEboX2A1-/view?usp=sharing) — 1,997,013 bytes.

**Option A — download inside the runtime:** run the cell below, then run the notebook top to bottom.

**Option B — upload from your machine:** download the file in your browser, then in Colab open the **Files** panel (left sidebar) → **⋮** → **Upload to notebook** → pick the file. It lands in the runtime's home folder. Rename it to `counts_annotated.tsv` if your browser saved it as `counts_annotated (1).tsv`.

Either way the cells below run unchanged — they read the file by name, no path.

In [1]:
# Option A: fetch the count matrix into the runtime
download.file("https://drive.google.com/uc?export=download&id=1Ym9S2vvbuTbAWq2mKFlQfLxLEboX2A1-",
              "counts_annotated.tsv")

cat("bytes:", file.info("counts_annotated.tsv")$size, "\n")
check <- read.delim("counts_annotated.tsv", row.names = 1, check.names = FALSE)
cat("genes:", nrow(check), "columns:", ncol(check), "\n")
cat("samples:", colnames(check), "\n")

bytes: 1997013 
genes: 39376 columns: 13 
samples: Symbol PC9_Control_1 PC9_Control_2 PC9_Control_3 PC9_Apatinib_1 PC9_Apatinib_2 PC9_Apatinib_3 PC9GR_Control_1 PC9GR_Control_2 PC9GR_Control_3 PC9GR_Apatinib_1 PC9GR_Apatinib_2 PC9GR_Apatinib_3 


## Statistics part

## 2. Design and unit of analysis

Write down the design, the unit of analysis, and the sample size **before** any test.

- The unit of analysis here is the **sample** — not the cell, not the gene, not the pixel in the image.

In [ ]:
counts <- read.delim("counts_annotated.tsv", row.names = 1, check.names = FALSE)
counts_mat <- as.matrix(counts[, -1])

cat("genes:", nrow(counts_mat), "samples:", ncol(counts_mat), "\n")
cat("total UMI per sample:\n")
print(colSums(counts_mat))
cat("span:", max(colSums(counts_mat)) - min(colSums(counts_mat)), "=",
    round(100 * (max(colSums(counts_mat)) - min(colSums(counts_mat))) / mean(colSums(counts_mat)), 1),
    "% of mean\n")


In [ ]:
samples <- colnames(counts)[-1]

sample_design <- data.frame(
  sample    = samples,
  cell_line = sub("_.*", "", samples),
  treatment = sub("^[^_]+_([^_]+)_.*", "\\1", samples),
  replicate = as.integer(sub(".*_", "", samples))
)

sample_design


What the design says:

- **n = 3** per group. Every comparison is 3 samples vs 3 samples.
- **12 samples total.** The 39,376 gene rows are measurements *on* those 12 samples, not 39,376 samples.
- **Two cell lines, two treatments.** A comparison must name the line: apatinib vs control is computed inside `PC9`, and separately inside `PC9GR`.
- **Library sizes span 2,996,230 UMI = 13.9 % of the mean.** A gene that looks "higher" in the 22,481,285-UMI sample than in the 19,485,055-UMI sample can be pure sequencing depth. Depth is a confounder in everything below.

Testing one gene once = 3 samples vs 3 samples. The same test runs 17,860 times (one per gene), so 17,860 hits are **not** 17,860 discoveries. The filter that sets the 17,860:

In [ ]:
keep_genes <- rowSums(counts_mat >= 10) >= 3
gene_counts <- counts_mat[keep_genes, ]
symbol_col <- counts$Symbol[keep_genes]
log_counts <- log2(gene_counts + 1)

cat("kept:", sum(keep_genes), "of", nrow(counts_mat), "\n")
cat("dropped:", sum(!keep_genes), "\n")
cat("zeros in the filtered matrix:", sum(gene_counts == 0),
    "of", length(gene_counts), "(", round(100 * mean(gene_counts == 0), 2), "%)\n")
cat("range of counts:", range(gene_counts), "\n")


## 3. Descriptive statistics

Describe the data before testing, and report the whole distribution — not one number.

In [ ]:
library_sizes <- colSums(counts_mat)

summary(library_sizes)
sd(library_sizes)
var(library_sizes)
IQR(library_sizes)


Library size (total UMI per sample):

- sd **970,172.5** UMI, IQR **872,587.2** UMI, variance 941,234,710,280.
- IQR < sd, so the spread comes from the tails: `PC9GR_Apatinib_3` (19,485,055) sits below the box, `PC9_Control_1` (22,481,285) above it.
- The mean is one number; the summary lines are the distribution.

In [ ]:
summary(log_counts[, "PC9_Control_1"])
IQR(log_counts[, "PC9_Control_1"])


For `PC9_Control_1`, per gene on the log2 scale:

- mean **7.841**, median **8.392** → mean **<** median.
- Raw counts of the same sample: mean **1,258**, median **335** → mean **>** median.
- The mean is pulled by a few huge genes. The median describes a typical gene.

In [ ]:
data.frame(
  raw_mean    = round(colMeans(gene_counts), 0),
  raw_median  = round(apply(gene_counts, 2, median), 0),
  log_mean    = round(colMeans(log_counts), 2),
  log_median  = round(apply(log_counts, 2, median), 2),
  log_sd      = round(apply(log_counts, 2, sd), 2),
  log_IQR     = round(apply(log_counts, 2, IQR), 2)
)


The pattern holds in **all 12 samples**: raw mean > median, log mean < median. `log2(count + 1)` is the deck's fix (3), applied to a real count matrix.

The matrix every test below runs on:

- **17,860 genes kept, 21,516 dropped.** Filter rule: count ≥10 in at least 3 samples.
- Counts run **0 to 751,956** UMI. There are 1,073 zeros out of 214,320 entries (0.5 %). A zero is a real measurement: the gene was not detected in that sample.
- `log2(count + 1)` keeps a zero as 0 (log of 0 is undefined) and shrinks the huge counts.

## 4. Distribution shape

Look at the shape before testing — the shape sets the limits of every test.

In [ ]:
sub5k <- log_counts[1:5000, ]

for (s in colnames(counts_mat)) {
  w <- shapiro.test(sub5k[, s])
  cat(s, "W =", round(w$statistic, 4), "p =", format(w$p.value, scientific = TRUE), "
")
}


Shapiro-Wilk caps at n = 5000, so each sample is tested on a 5,000-gene slice:

- **All 12 samples reject normality:** W 0.9741–0.9786, p 3.355658e-29 to 8.523301e-27.
- The count distribution is **not normal** in any sample.
- With n = 3 the test has no resolution: NCF4's three control values give W = 0.75, p < 2.2e-16.

In [ ]:
# Q-Q plot: the four extreme real samples, 5,000-gene slice
sub5k <- log_counts[1:5000, ]

panels <- c("PC9_Control_1", "PC9_Apatinib_1", "PC9GR_Control_1", "PC9GR_Apatinib_3")

png("qqplot.png", width = 900, height = 700)
par(mfcol = c(2, 2))
for (s in panels) {
  v <- sub5k[, s]
  qqnorm(v, main = s, xlab = "normal quantiles", ylab = "log2 count + 1")
  qqline(v, col = "#e7ad52", lwd = 2)
}
par(mfcol = c(1, 1))
dev.off()

for (s in panels) {
  q <- as.numeric(quantile(sub5k[, s], c(0.25, 0.5, 0.75)))
  cat(s, "Q1", round(q[1], 2), "med", round(q[2], 2), "Q3", round(q[3], 2),
      "left_gap", round(q[2] - q[1], 2), "right_gap", round(q[3] - q[2], 2),
      "zeros", sum(sub5k[, s] == 0), "\n")
}


All four panels show the same S-shape against the gold line:

- **Bottom:** zeros (14–39 genes per sample) sit flat at 0, far below the line. The lower quartile is 2.61–2.87 units below the median, while the middle gap is only 1.66–1.69.
- **Middle:** the curve follows the line.
- **Top:** points rise above the line at the right — the upper tail is heavier than a normal tail.

A bell-shaped model does not fit this data: too many zeros at the bottom, too heavy a tail at the top. The log transform helps, but it does not make the data normal — so (5) runs the rank tests too.

## 5. p-values

A p-value is the probability of seeing a difference this big **when there is no effect**. The null includes the spread, not just the gap.

Run: every filtered gene, PC9 control vs PC9 apatinib, Welch t-test on `log2(count + 1)`.

In [ ]:
ci <- which(sample_design$treatment == "Control"  & sample_design$cell_line == "PC9")
ai <- which(sample_design$treatment == "Apatinib" & sample_design$cell_line == "PC9")

t_safe <- function(v) {
  a <- v[ci]; b <- v[ai]
  if (sd(a) == 0 && sd(b) == 0) return(NA)
  if (sd(a) == 0 || sd(b) == 0) suppressWarnings(t.test(a, b)$p.value)
  t.test(a, b)$p.value
}
p_values <- apply(log_counts, 1, t_safe)

summary(p_values)
cat("raw hits:", sum(p_values < 0.05, na.rm = TRUE),
    "NA:", sum(is.na(p_values)), "of", length(p_values), "\n")
cat("expected by chance:", 17860 * 0.05, " raw hits minus expected:",
    sum(p_values < 0.05, na.rm = TRUE) - 17860 * 0.05, "\n")


The 17,860 p-values:

- min **0.0000009**, median **0.2315152**, **3,647** hits at p < 0.05, 10 NA.
- If no gene had an effect, the p-values would be spread evenly and 17,860 × 0.05 = **893** hits would appear by chance alone. So the honest reading of 3,647 raw hits is **2,754 above chance**.
- A median p of 0.2315152 says most genes show nothing.

17,860 tests is 17,860 chances to cross 0.05. Bonferroni and Benjamini–Hochberg (BH) on the real p-values:

In [ ]:
p_bonf <- p.adjust(p_values, method = "bonferroni")
p_bh   <- p.adjust(p_values, method = "BH")

cat("Bonferroni threshold:", 0.05 / 17860, "\n")
cat("Bonferroni hits:", sum(p_bonf < 0.05, na.rm = TRUE), "\n")
cat("BH hits:", sum(p_bh < 0.05, na.rm = TRUE), "\n")


- Bonferroni threshold: 0.05 / 17,860 = **2.799552e-06**. Bonferroni keeps **1** gene, BH keeps **2**.
- The 3,647 raw hits collapse to 1–2. 893 hits are expected by chance, so the raw list is signal + noise: the correction is what carries the result.

In [ ]:
pc9_res <- data.frame(
  Symbol   = symbol_col,
  log2FC   = round(rowMeans(log_counts[, ai]) - rowMeans(log_counts[, ci]), 2),
  mean_ctrl = round(rowMeans(gene_counts[, ci]), 0),
  mean_apo  = round(rowMeans(gene_counts[, ai]), 0),
  raw_p    = p_values,
  adj_p    = p_bh
)
head(pc9_res[order(p_bh), ], 6)


Two genes survive BH in PC9:

- **NCF4**: log2FC −1.04, 67 vs 32 UMI, raw p 8.605504e-07, adjusted **0.01536082**.
- **ANXA10**: log2FC −2.63, 233 vs 37 UMI, raw p 5.106063e-06, adjusted **0.04557162**.
- The next rows sit at adjusted p 0.09910547 (ENAH, NR4A2, EPHA4, WNT5A) — raw-p hits that BH rejects.

Report the adjusted p, the effect size, and the group means — not just "significant".

The p-value is driven by **spread**, not by the gap. Four real genes show it:

In [ ]:
gap <- rowMeans(log_counts[, ci]) - rowMeans(log_counts[, ai])
sd_max <- pmax(apply(log_counts[, ci], 1, sd), apply(log_counts[, ai], 1, sd))

for (g in c("NCF4", "FAM30A", "NUPR1", "PRSS2")) {
  i <- which(symbol_col == g)
  tt <- suppressWarnings(t.test(log_counts[i, ci], log_counts[i, ai]))
  cat(g, "gap", round(gap[i], 3), "ctrl", gene_counts[i, ci], "apo", gene_counts[i, ai],
      "| logmean", round(mean(log_counts[i, ci]), 4), round(mean(log_counts[i, ai]), 4),
      "| sd", round(sd(log_counts[i, ci]), 4), round(sd(log_counts[i, ai]), 4),
      "| t", round(tt$statistic, 3), "df", round(tt$parameter, 4),
      "| p", format(tt$p.value, scientific = TRUE), "\n")
}

p_ncf4 <- suppressWarnings(t.test(log_counts[which(symbol_col == "NCF4"), ci], log_counts[which(symbol_col == "NCF4"), ai]))$p.value
p_nupr1 <- suppressWarnings(t.test(log_counts[which(symbol_col == "NUPR1"), ci], log_counts[which(symbol_col == "NUPR1"), ai]))$p.value
cat("NUPR1 p / NCF4 p =", round(p_nupr1 / p_ncf4), "\n")


Same matrix, same n = 3, same test:

- **NCF4** — gap 1.036, sd 0.0245 and 0.0249 → t 51.386, p **8.605504e-07**. Small gap, almost no spread → tiny p.
- **NUPR1** — gap −3.044, bigger than NCF4's, but sd 2.1817 → t −2.375, p **0.1324607**. Bigger gap, big spread → not significant. NUPR1's p is **153,926×** weaker than NCF4's.
- **FAM30A** — sd 1.9179 → p **0.03144118**, barely significant, df collapsed to 2.469.
- **PRSS2** — gap −3.391, sd 1.3687 → p **0.02970003**.

The p follows the sd, not the gap.

N sets the power. n = 3 per group is the limit of every test here.

In [ ]:
sd_ctrl <- apply(log_counts[, ci], 1, sd)
sd_apo  <- apply(log_counts[, ai], 1, sd)

cat("sd == 0 in control:", sum(sd_ctrl == 0), "\n")
cat("sd == 0 in apatinib:", sum(sd_apo == 0), "\n")
cat("sd == 0 in both:", sum(sd_ctrl == 0 & sd_apo == 0), "\n")
cat("smallest possible Wilcoxon p at 3v3: 1 / choose(6, 3) =", 1 / choose(6, 3), "\n")
cat("floor at 5v5: 1 / choose(10, 5) =", 1 / choose(10, 5), "\n")

n4 <- which(symbol_col == "NCF4")
print(shapiro.test(log_counts[n4, ci]))
print(shapiro.test(log_counts[n4, ai]))

w_pc9 <- apply(log_counts, 1, function(v)
  tryCatch(suppressWarnings(wilcox.test(v[ci], v[ai]))$p.value, error = function(e) NA))
cat("Wilcoxon hits:", sum(w_pc9 < 0.05, na.rm = TRUE),
    "min:", min(w_pc9, na.rm = TRUE), "median:", median(w_pc9, na.rm = TRUE),
    "NA:", sum(is.na(w_pc9)), "\n")


- **Zero variance:** 59 genes are flat in the control group, 62 in the apatinib group, 10 in both. The 10 are the NA in the t-test cell: no spread means nothing to divide by, so those genes can never be significant.
- **Rank-test floor:** with 3 vs 3 samples the smallest Wilcoxon p is 1 / choose(6, 3) = **0.05**, so no gene can beat 0.05. At 5 vs 5 the floor is 1 / choose(10, 5) = 0.003968254 — resolution comes from n, not from the choice of test.
- **Real result:** Wilcoxon on this matrix gives **0 hits** (min 0.05934644, median 0.3758251, NA 10).
- **Shapiro at n = 3:** NCF4's control values (sd 0.0245) give W = 0.75, p < 2.2e-16 — "not normal" from 3 numbers.

Pooling raises n and lowers the variance, so the test looks stronger. Before pooling, check that the pooled groups are the same experiment. Pool the two cell lines into one 6 vs 6 treatment comparison:

In [ ]:
ctrl_all <- which(sample_design$treatment == "Control")
apo_all  <- which(sample_design$treatment == "Apatinib")

p_pool <- apply(log_counts, 1, function(v)
  tryCatch(suppressWarnings(t.test(v[ctrl_all], v[apo_all]))$p.value, error = function(e) NA))

cat("pooled 6v6 raw:", sum(p_pool < 0.05, na.rm = TRUE),
    "BH:", sum(p.adjust(p_pool, "BH") < 0.05, na.rm = TRUE), "\n")
cat("PC9 3v3 BH:", sum(p_bh < 0.05, na.rm = TRUE), "\n")


In [ ]:
grc <- which(sample_design$treatment == "Control"  & sample_design$cell_line == "PC9GR")
gra <- which(sample_design$treatment == "Apatinib" & sample_design$cell_line == "PC9GR")

p_gr <- apply(log_counts, 1, function(v)
  tryCatch(suppressWarnings(t.test(v[grc], v[gra]))$p.value, error = function(e) NA))
p_bh_gr <- p.adjust(p_gr, method = "BH")

cat("PC9GR raw:", sum(p_gr < 0.05, na.rm = TRUE), "BH:", sum(p_bh_gr < 0.05, na.rm = TRUE), "\n")

pc9gr_res <- data.frame(
  Symbol = symbol_col,
  log2FC = round(rowMeans(log_counts[, gra]) - rowMeans(log_counts[, grc]), 2),
  raw_p  = p_gr,
  adj_p  = p_bh_gr
)
head(pc9gr_res[order(p_bh_gr), ], 6)

fc_pc9 <- rowMeans(log_counts[, ai]) - rowMeans(log_counts[, ci])
fc_gr  <- rowMeans(log_counts[, gra]) - rowMeans(log_counts[, grc])
bh_pool <- p.adjust(p_pool, "BH") < 0.05
hits <- which(bh_pool)

cat("pooled BH hits moving the same direction in both lines:",
    sum(sign(fc_pc9[hits]) == sign(fc_gr[hits])), "of", length(hits), "\n")
cat("pooled BH hits that are raw-p hits in PC9 alone:", sum(p_values[hits] < 0.05, na.rm = TRUE), "\n")
cat("pooled BH hits that are raw-p hits in PC9GR alone:", sum(p_gr[hits] < 0.05, na.rm = TRUE), "\n")
cat("pooled BH hits that are raw-p hits in BOTH lines:",
    sum(p_values[hits] < 0.05 & p_gr[hits] < 0.05, na.rm = TRUE), "\n")


- **Pooled 6 vs 6:** 3,106 raw hits, **267 BH hits**. Within-line 3 vs 3: PC9 **2**, PC9GR **374**. Pooling halves the standard error, so it finds far more than either line alone.
- **Direction check:** all **267 of 267** pooled hits move the same way in both lines.
- **Overlap check:** the pooled hits share **0** genes with the 9,998 cell-line BH hits in (6).
- Both checks pass, so pooling is the right move here: apatinib moves 267 genes consistently. 229 are raw-p hits in PC9 alone, 244 in PC9GR alone, 206 in both.
- What pooling cannot say: which line did the moving.

## 6. Multi-group, categorical, correlation

ANOVA for 3+ groups (Kruskal-Wallis is the rank version), chi-square/Fisher for categories, correlation and regression for continuous pairs. This matrix has 4 groups: 2 lines × 2 treatments.

In [ ]:
anova_p <- function(v, f) tryCatch(anova(lm(v ~ f))[["Pr(>F)"]][1], error = function(e) NA)

group4 <- factor(paste(sample_design$cell_line, sample_design$treatment, sep = "_"))
p_grp4 <- apply(log_counts, 1, function(v) anova_p(v, group4))
p_kw4  <- apply(log_counts, 1, function(v)
  tryCatch(suppressWarnings(kruskal.test(v ~ group4))$p.value, error = function(e) NA))
p_line <- apply(log_counts, 1, function(v) anova_p(v, factor(sample_design$cell_line)))

cat("4-group ANOVA raw:", sum(p_grp4 < 0.05, na.rm = TRUE),
    "BH:", sum(p.adjust(p_grp4, "BH") < 0.05, na.rm = TRUE), "\n")
cat("cell-line ANOVA raw:", sum(p_line < 0.05, na.rm = TRUE),
    "BH:", sum(p.adjust(p_line, "BH") < 0.05, na.rm = TRUE), "\n")
cat("4-group Kruskal raw:", sum(p_kw4 < 0.05, na.rm = TRUE),
    "BH:", sum(p.adjust(p_kw4, "BH") < 0.05, na.rm = TRUE),
    "min p:", min(p_kw4, na.rm = TRUE), "min p x 17860 =", min(p_kw4, na.rm = TRUE) * 17860, "\n")


- **Four-group ANOVA:** 13,374 raw hits, **12,821 BH hits**. "12,821 genes differ among the four groups" is not an apatinib effect.
- **Cell line only:** 10,929 raw hits, **9,998 BH hits** — most of the four-group result is PC9 vs PC9GR.
- **Kruskal-Wallis:** 10,783 raw hits, **0 BH hits**, min p 0.01351344. BH multiplies the smallest p by 17,860 / 1 = 241.3501, which is above 1, so nothing survives.
- ANOVA needs normal data and equal variances; (4) showed the data is not normal. With 4 groups of 3, the rank test has no resolution. Both answers are weak, and n = 3 is why.

Chi-square compares what you counted with what independence predicts. Categories: "high in PC9" and "high in PC9GR", where **high = mean UMI over that line's 3 samples > 8**:

In [ ]:
high_pc9   <- rowMeans(gene_counts[, which(sample_design$cell_line == "PC9")]) > 8
high_pc9gr <- rowMeans(gene_counts[, which(sample_design$cell_line == "PC9GR")]) > 8

tab <- matrix(c(sum(high_pc9 & high_pc9gr), sum(high_pc9 & !high_pc9gr),
               sum(!high_pc9 & high_pc9gr), sum(!high_pc9 & !high_pc9gr)), nrow = 2)
tab
round(chisq.test(tab)$expected, 2)
chisq.test(tab)


In [ ]:
sig <- !is.na(p_bh) & p_bh < 0.05

tab2 <- matrix(c(sum(sig & high_pc9), sum(sig & !high_pc9),
               sum(!sig & high_pc9), sum(!sig & !high_pc9)), nrow = 2)
tab2
round(chisq.test(tab2)$expected, 2)
suppressWarnings(chisq.test(tab2))
fisher.test(tab2)


Concordance (mean UMI > 8 per line):

- **16,253** genes high in both lines, 685 high in PC9 only, 823 high in PC9GR only, 99 high in neither.
- Expected under independence: 16,194.47, 881.53, 743.53, 40.47. The high/high cell and the low/low cell are both above expectation.
- X² = **91.751**, p < 2.2e-16: the two cell lines share most of their expression.

Small counts (BH-significant genes × expression) give the table 2 / 0 / 16,936 / 922, so expected counts are 1.9 and 0.1:

- Chi-square warns "Chi-squared approximation may be incorrect" and returns p = 1.
- Expected counts below 5 → use Fisher's exact test. Fisher returns p = 1, odds ratio Inf (CI 0.01019845 to Inf).
- p = 1 is the largest possible p: the table is fully compatible with independence, so there is no association to report. 2 significant genes out of 17,860 carry no information about expression level.

Pearson measures a straight-line relationship, Spearman measures any monotone one. Two real gene rows across the 12 samples:

In [ ]:
a <- gene_counts[which(symbol_col == "AGPAT5"), ]
b <- gene_counts[which(symbol_col == "NCF4"), ]
a
b

cor.test(a, b)
suppressWarnings(cor.test(a, b, method = "spearman"))


- AGPAT5 runs 371–755 UMI in every sample; NCF4 runs 18–68.
- Pearson r = **0.9632051**, p 4.993e-07. Spearman rho = **0.968427**, p 2.343e-07. r ≈ 0.96 is strong, the relationship is monotone, so both tests agree.
- n = 12 points: a big r is significant, a modest r on the same 12 points is not.

Look for the confounder. In this matrix the confounder is sequencing depth.

In [ ]:
depth <- colSums(counts_mat)
cat("AGPAT5 vs depth r:", round(cor(a, depth), 7), "\n")
cat("NCF4  vs depth r:", round(cor(b, depth), 7), "\n")


In [ ]:
a_hot <- a
a_hot[which.max(a_hot)] <- a_hot[which.max(a_hot)] * 10
cat(names(a_hot[which.max(a_hot)]), a_hot[which.max(a_hot)], "\n")

cor.test(a_hot, b)
suppressWarnings(cor.test(a_hot, b, method = "spearman"))


- Depth correlates with both genes: r **0.6139899** (AGPAT5) and **0.6755128** (NCF4). Part of any gene-gene correlation in this matrix is shared depth.
- **Hot-spot test:** multiply the largest AGPAT5 value (`PC9_Control_2`, 755 → 7,550). Pearson drops to **0.4863409**, p **0.1089** — significance gone. Spearman is unchanged: rho 0.968427, p 2.343e-07.
- One point out of 12 breaks Pearson. Ranks ignore size.

In [ ]:
fit <- lm(b ~ a)
fit
summary(fit)


- Intercept **−48.07464**, slope **0.15633**, R² **0.9278**, F 128.4 on 1 and 10 DF, p 4.993e-07, residual SE 5.517.
- The negative intercept means the line predicts NCF4 = −9 when AGPAT5 = 250 — a count cannot be negative.
- The fit describes the observed range (371 to 755 UMI). It is not valid outside it.

Correlate the samples with each other. All 39,376 genes, then the filtered matrix:

In [ ]:
round(cor(log2(counts_mat + 1)), 3)
round(cor(log_counts), 3)


- All genes: within PC9 0.989–0.993, within PC9GR 0.985–0.993, across lines 0.969–0.980.
- Filtered: within PC9 0.980–0.993, within PC9GR 0.965–0.992, across lines 0.899–0.943.
- No outlier sample: min 0.899, max 0.993. The cross-line drop is the cell-line effect.
- High correlations mean the **samples** are similar, not that the groups are. Every gene row shares the same 12 columns, so all 17,860 tests share the same depth, library, and batch.

In [ ]:
pc9_out <- pc9_res[order(p_bh), ]
write.csv(pc9_out[pc9_out$adj_p < 0.05, ], "pc9_apatinib_results.csv", row.names = FALSE)
cat("wrote", sum(pc9_out$adj_p < 0.05, na.rm = TRUE), "rows to pc9_apatinib_results.csv\n")


## Graphics part

## 7. From numbers to figures — real data

- **Volcano:** one dot per gene. x = effect (log2FC), y = −log10(BH-adjusted p).
- **Heatmap:** one cell per gene × sample. Colour = log2(count + 1).
- No new data, no new tests — only the 17,860 genes and 2 BH hits computed above.

In [ ]:
cat("genes:", sum(!is.na(pc9_res$adj_p)), " BH hits:", sum(pc9_res$adj_p < 0.05, na.rm = TRUE), " NA:", sum(is.na(pc9_res$adj_p)), "\n")
print(pc9_out[1:6, ])
plot(pc9_res$log2FC[!is.na(pc9_res$adj_p)], -log10(pc9_res$adj_p[!is.na(pc9_res$adj_p)]), pch = 16, cex = 0.6, col = "grey30", xlab = "log2FC (Apatinib - Control)", ylab = "-log10(BH adj p)")
head_i <- pc9_res$adj_p < 0.05
points(pc9_res$log2FC[head_i], -log10(pc9_res$adj_p[head_i]), pch = 16, col = "red")
abline(h = -log10(0.05), v = c(-1, 1), lty = 2, col = "red")
text(x = pc9_res$log2FC[head_i], y = -log10(pc9_res$adj_p[head_i]), labels = pc9_res$Symbol[head_i], pos = c(2, 2), col = "red")

- 17,850 plotted genes: 10 NA (both groups sd = 0) dropped.
- Above the dashed line (FDR 0.05): 2 games only — **NCF4** (log2FC −1.04, adj_p 0.01536082) and **ANXA10** (log2FC −2.63, adj_p 0.04557162), top left (both go down in Apatinib).
- Next in line sits at adj_p 0.09910547 (ENAH, NR4A2, EPHA4, WNT5A) — just below the line, raw hits BH rejects.
- The vast cloud sits near x = 0, y = 0: most genes do not move.

In [ ]:
top6 <- pc9_res[order(pc9_res$adj_p), ][1:6, ]
print(top6)
heatmap_mat <- log_counts[match(top6$Symbol, symbol_col), c(ci, ai)]
print(round(heatmap_mat, 2))
heatmap(heatmap_mat, Rowv = NA, Colv = NA, scale = "none", margins = c(8, 8), main = "Top 6 PC9 Apatinib genes")

- 6 rows × 6 columns: the 6 smallest adj_p, PC9 Control ×3 then Apatinib ×3.
- NCF4 ~6.1 vs ~5.0; ANXA10 ~7.9 vs ~5.2 — dark on the left, pale on the right (down in Apatinib).
- ENAH ~13.0 vs ~12.7 and WNT5A ~12.3 vs ~11.3 — same direction, smaller move.
- Colour is log2(count + 1); the same 6 rows are the red dots on the volcano's left edge.

## The report

- **Design (2):** 2 lines × 2 treatments × 3 replicates = 12 samples, n = 3 per group; library sizes span 13.9 % of the mean.
- **Descriptive (3):** raw mean > median in all 12 samples, log mean < median in all 12; counts 0 to 751,956, 1,073 zeros of 214,320 (0.5 %).
- **Shape (4):** not normal in all 12 samples (W 0.9741–0.9786, p 3.355658e-29 to 8.523301e-27). Q-Q: zeros below the line, middle on the line, heavy tail above.
- **p-values (5):** 17,860 t-tests in PC9 → 3,647 raw hits, 893 expected by chance, median p 0.2315152. Bonferroni keeps 1, BH keeps 2: NCF4, ANXA10.
- **Spread (5):** NCF4 (sd 0.0245, p 8.605504e-07) vs NUPR1 (gap −3.044, sd 2.1817, p 0.1324607). The p follows the sd.
- **n (5):** Wilcoxon floor 1 / choose(6, 3) = 0.05 → 0 hits (min 0.05934644). 59 genes flat in control, 10 in both.
- **Pooling (5):** pooled 6v6 → 267 BH hits, all 267 same direction in both lines, 0 overlap with the 9,998 cell-line hits; within-line 2 (PC9), 374 (PC9GR).
- **ANOVA / Kruskal (6):** four-group ANOVA 12,821 BH hits, cell line alone 9,998; Kruskal 0 BH hits, min p 0.01351344.
- **Categories (6):** concordance X² = 91.751, p < 2.2e-16. The BH-hit × expression table is too small — Fisher p = 1.
- **Correlation (6):** AGPAT5 vs NCF4 r 0.9632051, Spearman 0.968427; one hot spot breaks Pearson (0.4863409, p 0.1089), not Spearman; depth 0.6139899 / 0.6755128; samples 0.899–0.993.

Bottom line: cell line, library depth, and n = 3 dominate this matrix. The surviving apatinib finding in PC9 is 2 genes.